<a href="https://colab.research.google.com/github/nivedita-rmsh/CLED/blob/main/Translation_to_French.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import json
from collections import Counter

Verifying existence of MAVEN files

In [ ]:
maven_dir = '/content/drive/MyDrive/NLP_data'
for f in os.listdir(maven_dir):
    size = os.path.getsize(os.path.join(maven_dir, f)) / 1024
    print(f"{f:40s} {size:.1f} KB")

Load and Inspect Structure of JSON files

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train = load_jsonl(f'{maven_dir}/train.jsonl')

# Look at one document
doc = train[0]
print("Keys in a document:", list(doc.keys()))
print("Title:", doc['title'])
print("Number of sentences:", len(doc['content']))
print("Number of events:", len(doc['events']))

Looking into sentences with their event triggers highlighted.

In [ ]:
for doc in train[:3]:
    print(f"\n=== {doc['title']} ===")
    for event in doc['events'][:2]:
        etype = event['type']
        for mention in event['mention'][:1]:
            sid   = mention['sent_id']
            start, end = mention['offset']
            tokens = doc['content'][sid]['tokens']
            trigger = ' '.join(tokens[start:end])
            sentence = ' '.join(tokens)
            print(f"  Event type : {etype}")
            print(f"  Trigger    : '{trigger}'  (tokens {start}–{end})")
            print(f"  Sentence   : {sentence}\n")

Event Stats

In [ ]:
event_types = Counter()
total_mentions = 0

for doc in train:
    for event in doc['events']:
        event_types[event['type']] += len(event['mention'])
        total_mentions += len(event['mention'])

print(f"Total documents   : {len(train)}")
print(f"Total event types : {len(event_types)}")
print(f"Total mentions    : {total_mentions}")
print(f"\nTop 10 event types:")
for etype, count in event_types.most_common(10):
    print(f"  {etype:30s} {count}")

Convert to BIO format

In [ ]:
def doc_to_bio_examples(doc):
    examples = []
    trigger_map = {}
    for event in doc['events']:
        for mention in event['mention']:
            sid = mention['sent_id']
            trigger_map.setdefault(sid, []).append(
                (mention['offset'][0], mention['offset'][1], event['type'])
            )

    for sid, sent in enumerate(doc['content']):
        tokens = sent['tokens']
        labels = ['O'] * len(tokens)
        for start, end, etype in trigger_map.get(sid, []):
            for i in range(start, end):
                labels[i] = f"B-{etype}" if i == start else f"I-{etype}"
        examples.append({'tokens': tokens, 'labels': labels, 'doc_id': doc['id'], 'sent_id': sid})
    return examples

# Test it
sample = doc_to_bio_examples(train[0])
for ex in sample[:3]:
    pairs = list(zip(ex['tokens'], ex['labels']))
    non_o = [(t, l) for t, l in pairs if l != 'O']
    if non_o:
        print(ex['tokens'])
        print(ex['labels'])
        print()

Sanity Check

In [ ]:
def check_bio_integrity(examples):
    errors = 0
    for ex in examples:
        prev = 'O'
        for i, label in enumerate(ex['labels']):
            if label.startswith('I-'):
                etype = label[2:]
                if prev != f'B-{etype}' and prev != f'I-{etype}':
                    print(f"BIO error at token {i}: '{label}' follows '{prev}'")
                    print(f"  Tokens: {ex['tokens']}")
                    errors += 1
            prev = label
    print(f"\nTotal BIO errors: {errors}")

check_bio_integrity(sample)

Translation to FRENCH

In [ ]:
!pip install transformers sentencepiece sacremoses -q

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
import torch
import json
from tqdm import tqdm

In [ ]:
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer  = MarianTokenizer.from_pretrained(model_name)
model      = MarianMTModel.from_pretrained(model_name)

print("Model loaded successfully")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)

def translate_batch(sentences, batch_size=32, max_length=128):
    """
    Translate a list of English strings to French.
    Returns a list of translated strings, same length as input.
    """
    results = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        safe_batch = [s if s.strip() else "." for s in batch]

        inputs = tokenizer(
            safe_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            translated = model.generate(**inputs, max_length=max_length)

        decoded = tokenizer.batch_decode(translated, skip_special_tokens=True)
        results.extend(decoded)

    return results

In [ ]:

def translate_maven_split(input_path, output_path):
    """
    Reads a MAVEN .jsonl file, translates all sentence text to French,
    writes a new .jsonl with translated content alongside the original.
    """
    # Load all documents
    with open(input_path) as f:
        docs = [json.loads(line) for line in f]

    # Collect all sentences across all docs for batch translation
    # We track (doc_idx, sent_idx) so we can put them back
    all_sentences = []
    index_map     = []   # (doc_idx, sent_idx)

    for d_idx, doc in enumerate(docs):
        for s_idx, sent in enumerate(doc['content']):
            all_sentences.append(sent['sentence'])
            index_map.append((d_idx, s_idx))

    print(f"Translating {len(all_sentences)} sentences from {len(docs)} documents...")

    translated = []
    batch_size = 32
    for i in tqdm(range(0, len(all_sentences), batch_size)):
        batch = all_sentences[i : i + batch_size]
        translated.extend(translate_batch(batch, batch_size=batch_size))

    # Put translated sentences back into the document structure
    for (d_idx, s_idx), fr_sentence in zip(index_map, translated):
        docs[d_idx]['content'][s_idx]['sentence_fr'] = fr_sentence
        # Keep original sentence too — useful for debugging alignment

    with open(output_path, 'w') as f:
        for doc in docs:
            f.write(json.dumps(doc, ensure_ascii=False) + '\n')

    print(f"Done. Saved to {output_path}")

In [ ]:
maven_dir = '/content/drive/MyDrive/NLP_data'

translate_maven_split(
    f'{maven_dir}/train.jsonl',
    f'{maven_dir}/train_fr.jsonl'
)

translate_maven_split(
    f'{maven_dir}/valid.jsonl',
    f'{maven_dir}/valid_fr.jsonl'
)

translate_maven_split(
    f'{maven_dir}/test.jsonl',
    f'{maven_dir}/test_fr.jsonl'
)

In [ ]:
with open(f'{maven_dir}/train_fr.jsonl') as f:
    docs_fr = [json.loads(line) for line in f]

doc = docs_fr[0]
print(f"Document: {doc['title']}\n")

for sent in doc['content'][:4]:
    print(f"EN: {sent['sentence']}")
    print(f"FR: {sent['sentence_fr']}")
    print()